In [3]:
import math 
import time 

import spacy
import torch
import torchtext

from torch import nn, optim 
from torch.optim import Adam 
from torch import Tensor 

/data/anaconda3/envs/rosellm/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
print(device)

cpu


In [6]:
batch_size = 128  # How many sentences in a batch.
max_len = 256  # Maximum length of a sentence.

### Model parameters.

d_model = 512  # Word embedding size.
n_heads = 8  # Number of attention heads.

# Given d_model = 512 
# Given n_heads = 8 
# We have d_k = d_v = d_model / n_heads = 512 / 8 = 64
# which is the dimension of Q, K, V.

n_layers = 6  # Number of layers in the encoder/decoder.
d_ff = 2048  # Dimension of the feedforward network.

# 512 => 2048 => 512
# d_model => d_ff => d_model
# d_ff is also called "projection size".

n_hidden = d_ff  # Number of hidden units in the feedforward network.

dropout = 0.1  # Dropout probability.


### Optimizer parameters.

init_lr = 1e-5  # Initial learning rate.
factor = 0.9  # Factor by which the learning rate will be reduced.
adam_eps = 5e-9  # Adam epsilon.
patience = 10  # Patience for early stopping.
warmup = 100  # Number of steps for the warmup phase.
epoch = 100  # Number of epochs to train.
clip = 1.0  # Gradient clipping.
weight_decay = 5e-4  # Weight decay.
inf = float("inf")  # Infinity.


In [7]:
class Tokenizer:
    def __init__(self):
        self.spacy_de = spacy.load("de_core_news_sm")
        self.spacy_en = spacy.load("en_core_web_sm")

    def tokenize_de(self, text):
        return [tok.text for tok in self.spacy_de.tokenizer(text)]
    
    def tokenize_en(self, text):
        return [tok.text for tok in self.spacy_en.tokenizer(text)]
    
tokenizer = Tokenizer() 
example = 'This is an example sentence.'
tokens = tokenizer.tokenize_en(example)
print(example)
print(tokens)


This is an example sentence.
['This', 'is', 'an', 'example', 'sentence', '.']


In [8]:
example = 'two yong, white males are outside near many bushes'
tokens = tokenizer.tokenize_de(example)
print(example)
print(tokens)


two yong, white males are outside near many bushes
['two', 'yong', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes']


In [9]:
from torchtext.data import Field, BucketIterator
from torchtext.datasets import Multi30k
import urllib.request
import os
import gzip

class DataLoader:
    source: Field = None
    target: Field = None
    def __init__(self, ext, tokenize_en, tokenize_de, init_token, eos_token):
        self.ext = ext
        self.tokenize_en = tokenize_en
        self.tokenize_de = tokenize_de
        self.init_token = init_token
        self.eos_token = eos_token
        print(f"Initializing DataLoader with extension: {ext}")
        
    def download_dataset(self):
        """Download Multi30k dataset files from GitHub"""
        data_dir = '.data/multi30k'
        os.makedirs(data_dir, exist_ok=True)
        
        base_url = 'https://github.com/multi30k/dataset/raw/master/data/task1/raw'
        files = {
            'train': 'train',
            'val': 'val',
            'test2016': 'test_2016_flickr'  # Map test2016 to correct filename
        }
        
        for local_name, remote_name in files.items():
            for lang in ['en', 'de']:
                url = f'{base_url}/{remote_name}.{lang}.gz'
                compressed_path = f'{data_dir}/{local_name}.{lang}.gz'
                final_path = f'{data_dir}/{local_name}.{lang}'
                
                if not os.path.exists(final_path):
                    print(f'Downloading {url} to {compressed_path}')
                    urllib.request.urlretrieve(url, compressed_path)
                    
                    # Decompress .gz file
                    with gzip.open(compressed_path, 'rb') as f_in:
                        with open(final_path, 'wb') as f_out:
                            f_out.write(f_in.read())
                    
                    # Remove compressed file
                    os.remove(compressed_path)
    
    def make_dataset(self):
        # Download dataset files
        self.download_dataset()
        
        if self.ext == ('.de', '.en'):
            self.source = Field(tokenize=self.tokenize_de, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
            self.target = Field(tokenize=self.tokenize_en, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
        elif self.ext == ('.en', '.de'):
            self.source = Field(tokenize=self.tokenize_en, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
            self.target = Field(tokenize=self.tokenize_de, init_token=self.init_token, eos_token=self.eos_token, lower=True, batch_first=True)
        else:
            raise ValueError(f"Extension {self.ext} not supported.")
            
        train_data, valid_data, test_data = Multi30k.splits(
            exts=self.ext,
            fields=(self.source, self.target), 
            root='.data'
        )
        return train_data, valid_data, test_data
    
    def build_vocab(self, train_data, min_freq=2):
        self.source.build_vocab(train_data, min_freq=min_freq)
        self.target.build_vocab(train_data, min_freq=min_freq)

    def make_iter(self, train_data, valid_data, test_data, batch_size, device):
        train_iterator, valid_iterator, test_iterator = BucketIterator.splits(
            (train_data, valid_data, test_data),
            batch_size=batch_size,
            device=device
        )
        print(f"Train size: {len(train_iterator)}")
        print(f"Valid size: {len(valid_iterator)}")
        print(f"Test size: {len(test_iterator)}")
        print("Initialization done.")
        return train_iterator, valid_iterator, test_iterator

data_loader = DataLoader(
    ext=('.de', '.en'),
    tokenize_en=tokenizer.tokenize_en,
    tokenize_de=tokenizer.tokenize_de,
    init_token='<sos>',
    eos_token='<eos>'
)
train_data, valid_data, test_data = data_loader.make_dataset()
data_loader.build_vocab(train_data)
train_iterator, valid_iterator, test_iterator = data_loader.make_iter(train_data, valid_data, test_data, batch_size, device)

Initializing DataLoader with extension: ('.de', '.en')
Train size: 227
Valid size: 8
Test size: 8
Initialization done.


In [10]:
print(train_data.examples[0].src)
print(train_data.examples[0].trg)


['zwei', 'junge', 'weiße', 'männer', 'sind', 'im', 'freien', 'in', 'der', 'nähe', 'vieler', 'büsche', '.']
['two', 'young', ',', 'white', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [11]:
print("1. check vocabulary size")
print("source vocab size: ", len(data_loader.source.vocab))
print("target vocab size: ", len(data_loader.target.vocab))

1. check vocabulary size
source vocab size:  7853
target vocab size:  5893


In [12]:
print("2. check mapping between words and token indices")
print('word\t->\t token')
print("<sos>\t->\t ", data_loader.source.vocab.stoi['<sos>'])
print("<eos>\t->\t ", data_loader.source.vocab.stoi['<eos>'])
print("two\t->\t ", data_loader.target.vocab.stoi['two'])
print("many\t->\t ", data_loader.target.vocab.stoi['many'])
print("<pad>\t->\t ", data_loader.source.vocab.stoi['<pad>'])


2. check mapping between words and token indices
word	->	 token
<sos>	->	  2
<eos>	->	  3
two	->	  16
many	->	  202
<pad>	->	  1


In [13]:
print("3. get from iterator to select pairs")
# The function of <pad> is to pad the sentence to the same length.
# E.g. if the max length of sentence is l, then every sentence in the batch will be padded to l.
# My question: the max length of sentence is determined by the longest sentence in the training set or the max length of sentences in the batch?
# Answer: the max length of sentence is determined by the longest sentence in the batch.
for i, batch in enumerate(train_iterator):
    print(batch.src)
    print(batch.trg)
    break

3. get from iterator to select pairs
tensor([[   2,    8, 2229,  ...,    1,    1,    1],
        [   2,    5,  171,  ...,    1,    1,    1],
        [   2,   76, 2972,  ...,    1,    1,    1],
        ...,
        [   2,    5,  684,  ...,    1,    1,    1],
        [   2,   30,    7,  ...,    1,    1,    1],
        [   2,    8,   16,  ...,    1,    1,    1]])
tensor([[  2, 106,  14,  ...,   1,   1,   1],
        [  2,   4,  61,  ...,   1,   1,   1],
        [  2,   4, 573,  ...,   1,   1,   1],
        ...,
        [  2, 106, 154,  ...,   1,   1,   1],
        [  2,  30,  22,  ...,   1,   1,   1],
        [  2,   4,  14,  ...,   1,   1,   1]])


In [14]:
print("4. the following idices are also very important")
src_pad_idx = data_loader.source.vocab.stoi['<pad>']
trg_pad_idx = data_loader.target.vocab.stoi['<pad>']
src_sos_idx = data_loader.source.vocab.stoi['<sos>']
trg_sos_idx = data_loader.target.vocab.stoi['<sos>']
print(f"Source PAD index: {src_pad_idx}")
print(f"Target PAD index: {trg_pad_idx}")
print(f"Source SOS index: {src_sos_idx}")
print(f"Target SOS index: {trg_sos_idx}")

4. the following idices are also very important
Source PAD index: 1
Target PAD index: 1
Source SOS index: 2
Target SOS index: 2


In [15]:
enc_voc_size = len(data_loader.source.vocab)
dec_voc_size = len(data_loader.target.vocab)
print(f"Encoder vocabulary size: {enc_voc_size}, as the dimension of the input parameter x in the embedding layer")
print(f"Decoder vocabulary size: {dec_voc_size}, as the dimension of the output parameter x in the fully connected layer")

Encoder vocabulary size: 7853, as the dimension of the input parameter x in the embedding layer
Decoder vocabulary size: 5893, as the dimension of the output parameter x in the fully connected layer


In [16]:
# Get data from iter.
# only run once to ensure the same data is used for testing.
for i, batch in enumerate(test_iterator):
    src = batch.src
    trg = batch.trg
    print("save src shape: ", src.shape)
    print("save trg shape: ", trg.shape)
    torch.save(src, f".data/tensor_src.pt")
    torch.save(trg, f".data/tensor_trg.pt")
    print(src)
    print(trg)
    break

test_src = torch.load(".data/tensor_src.pt")
test_trg = torch.load(".data/tensor_trg.pt")
print("shape of test_src: ", test_src.shape)
print("shape of test_trg: ", test_trg.shape)


save src shape:  torch.Size([128, 10])
save trg shape:  torch.Size([128, 14])
tensor([[   2,   18, 6787,  ...,  123,    4,    3],
        [   2,  105,   41,  ...,   91,    4,    3],
        [   2,    5,   26,  ..., 3449,    4,    3],
        ...,
        [   2,    8,   16,  ...,    1,    1,    1],
        [   2,   26,   16,  ...,    1,    1,    1],
        [   2,   18,   30,  ...,    1,    1,    1]])
tensor([[   2,   16, 1909,  ...,    1,    1,    1],
        [   2,  110,   19,  ...,    1,    1,    1],
        [   2,    4,   34,  ...,    1,    1,    1],
        ...,
        [   2,    4,   14,  ...,    1,    1,    1],
        [   2,   24,   14,  ...,    1,    1,    1],
        [   2,   16,   30,  ...,    1,    1,    1]])
shape of test_src:  torch.Size([128, 10])
shape of test_trg:  torch.Size([128, 14])


/tmp/ipykernel_1106594/284749364.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_src = torch.load(".data/tensor_src.pt")
/tmp/ipykernel_1106594/284749364.py:15: Fu

In [17]:
import nltk
# BLEU: Bilingual Evaluation Understudy

def calculate_bleu(reference, candidate):
    reference = [reference.split()]
    candidate = candidate.split()
    smoothing_function = nltk.translate.bleu_score.SmoothingFunction()
    bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate, smoothing_function=smoothing_function.method1)
    return bleu

reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is on the mat"
bleu_score = calculate_bleu(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")
candidate_sentence = "The cat is setting on the mat"
bleu_score = calculate_bleu(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")

BLEU score: 1.0
BLEU score: 0.274941620352113


In [22]:
from collections import Counter
from fractions import Fraction
import math

def calculate_bleu_detailed(reference, candidate):
    """Calculate BLEU score between reference and candidate sentences."""
    # Convert strings to lists of words
    reference = [reference.split()]  # Wrap in list since we expect list of references
    candidate = candidate.split()
    
    # Use default weights for BLEU-4 (equal weights)
    weights = (0.25, 0.25, 0.25, 0.25)
    
    # Calculate modified n-gram precisions for n=1,2,3,4
    precisions = []
    for n in range(1, 5):
        # Get n-grams for candidate and reference
        candidate_ngrams = Counter([tuple(candidate[i:i+n]) 
                                  for i in range(len(candidate)-n+1)]) if len(candidate) >= n else Counter()
        
        # Find max reference counts
        max_ref_counts = {}
        for ref in reference:
            ref_ngrams = Counter([tuple(ref[i:i+n]) 
                                for i in range(len(ref)-n+1)]) if len(ref) >= n else Counter()
            for ngram in candidate_ngrams:
                max_ref_counts[ngram] = max(max_ref_counts.get(ngram, 0), ref_ngrams[ngram])
        
        # Calculate clipped counts
        clipped_counts = {ngram: min(count, max_ref_counts.get(ngram, 0))
                         for ngram, count in candidate_ngrams.items()}
        
        numerator = sum(clipped_counts.values())
        denominator = sum(candidate_ngrams.values()) if candidate_ngrams else 1
        precisions.append(numerator / denominator if denominator else 0)
    
    # Calculate brevity penalty
    ref_len = len(reference[0])  # Use first reference length
    cand_len = len(candidate)
    bp = math.exp(1 - ref_len/cand_len) if cand_len < ref_len else 1
    
    # Calculate final BLEU score
    if precisions[0] == 0:
        return 0
    
    log_precisions = [math.log(p) if p > 0 else float('-inf') for p in precisions]
    weighted_score = sum(w * p for w, p in zip(weights, log_precisions))
    bleu = bp * math.exp(weighted_score)
    
    return bleu

In [23]:
reference_sentence = "The cat is on the mat"
candidate_sentence = "The cat is on the mat"
bleu_score = calculate_bleu_detailed(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")
candidate_sentence = "The cat is setting on the mat"
bleu_score = calculate_bleu_detailed(reference_sentence, candidate_sentence)
print(f"BLEU score: {bleu_score}")

BLEU score: 1.0
BLEU score: 0.0
